In [141]:
%restart_python

## Setup

In [153]:
import os
import sys

# Mitigates CUDA memory fragmentation -- must be set before torch import/any CUDA context exists.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import gc
import time
import torch
import numpy as np
import pandas as pd
import rdflib
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
from pykeen.models import TransE
from pykeen.training import SLCWATrainingLoop
from pykeen.sampling import BasicNegativeSampler

# Reused so TransE's train/val/test case split is identical to RGCN/graph_sage.ipynb's.
from training_data import Ontology, TrainingData

In [154]:
def construct_masked_tsv(schema_ttl_path: str, data_nt_path: str, output_tsv_path: str, exclusion_uri: str) -> None:
    """Merges schema + instance data into a masked TSV edge list, excluding the
    target-variable predicate. Cached at output_tsv_path -- delete manually to rebuild."""
    if os.path.exists(output_tsv_path):
        print(f"Found existing masked TSV at {output_tsv_path}. Skipping reconstruction...")
        return

    print(f"Constructing masked TSV artifact at {output_tsv_path}...")
    
    with open(output_tsv_path, 'w', encoding='utf-8') as outfile:
        
        schema_graph = rdflib.Graph()
        schema_graph.parse(schema_ttl_path, format="turtle")
        for subject, predicate, obj in schema_graph:
            s_str = subject.n3().replace('\t', ' ')
            p_str = predicate.n3().replace('\t', ' ')
            o_str = obj.n3().replace('\t', ' ')
            outfile.write(f"{s_str}\t{p_str}\t{o_str}\n")
            
        with open(data_nt_path, 'r', encoding='utf-8') as infile:
            for line in infile:
                
                if exclusion_uri in line:
                    continue
                    
                line = line.strip()
                if not line:
                    continue
                
                if line.endswith('.'):
                    line = line[:-1].strip()
                
                parts = line.split(' ', maxsplit=2)
                if len(parts) == 3:
                    s, p, o = parts
                    s = s.replace('\t', ' ')
                    p = p.replace('\t', ' ')
                    o = o.replace('\t', ' ')
                    
                    outfile.write(f"{s}\t{p}\t{o}\n")

# Same balanced-subset instance files RGCN.ipynb/graph_sage.ipynb/extract_graph_geometry.ipynb load.
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
target_exclusion_string = "/SurveyRating/"

sulo_schema_path = f"{volume}sulo_kg.ttl"
sulo_data_path = f"{volume}subgraph_balanced_sulo_populated_graph.nt"
sulo_merged_path = f"{volume}merged_sulo_training.nt"

baseline_schema_path = f"{volume}base_kg.ttl"
baseline_data_path = f"{volume}subset_balanced_baseline_populated_graph.nt"
baseline_merged_path = f"{volume}merged_baseline_training.nt"

transe_out_dir = f"{volume}transe/"
os.makedirs(transe_out_dir, exist_ok=True)

baseline_embeddings_path = f"{transe_out_dir}baseline_transe_embeddings.pt"
sulo_embeddings_path = f"{transe_out_dir}sulo_transe_embeddings.pt"

baseline_split_cache_path = f"{transe_out_dir}base_split.pt"
sulo_split_cache_path = f"{transe_out_dir}sulo_split.pt"
transe_results_csv_path = f"{transe_out_dir}transe_results.csv"

construct_masked_tsv(sulo_schema_path, sulo_data_path, sulo_merged_path, target_exclusion_string)
construct_masked_tsv(baseline_schema_path, baseline_data_path, baseline_merged_path, target_exclusion_string)

Found existing masked TSV at /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_sulo_training.nt. Skipping reconstruction...
Found existing masked TSV at /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_baseline_training.nt. Skipping reconstruction...

In [155]:
def build_case_split(nt_file_path, label_predicate, cache_path=None):
    """Builds the SAME 80/10/10 train/val/test Case split RGCN.ipynb/graph_sage.ipynb
    use (same random_state=42), returned as raw URI strings for direct use as
    entity_mapping keys. Cached at cache_path -- delete manually if
    nt_file_path/label_predicate change."""
    if cache_path and os.path.exists(cache_path):
        print(f"Found existing case split at {cache_path}. Loading instead of re-parsing {nt_file_path}...")
        return torch.load(cache_path, weights_only=False)

    ont = Ontology(nt_file_path=nt_file_path, label_predicate=label_predicate)
    idx_to_uri = {idx: uri for uri, idx in ont.enum_nodes.items()}

    split = TrainingData()
    split.create_training_data(ont)

    def uris_and_labels(x_idx, y_val):
        uris = [idx_to_uri[int(i)] for i in x_idx.tolist()]
        labels = y_val.numpy()
        return uris, labels

    result = {
        'train': uris_and_labels(split.x_train, split.y_train),
        'val': uris_and_labels(split.x_val, split.y_val),
        'test': uris_and_labels(split.x_test, split.y_test),
    }

    del ont, idx_to_uri, split  # frees the full edge_index/edge_type Ontology builds just to reach survey_labels
    gc.collect()

    if cache_path:
        print(f"Saving case split to {cache_path}...")
        torch.save(result, cache_path)

    return result


baseline_split = build_case_split(baseline_data_path, label_predicate="hasSurveyRating", cache_path=baseline_split_cache_path)
sulo_split = build_case_split(sulo_data_path, label_predicate="hasFeature", cache_path=sulo_split_cache_path)

for name, split in [('Ad-hoc (Base)', baseline_split), ('SULO', sulo_split)]:
    print(f"{name}: {len(split['train'][0])} train / {len(split['val'][0])} val / "
          f"{len(split['test'][0])} test cases")

Found existing case split at /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/transe/base_split.pt. Loading instead of re-parsing /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/subset_balanced_baseline_populated_graph.nt...
Found existing case split at /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/transe/sulo_split.pt. Loading instead of re-parsing /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/subgraph_balanced_sulo_populated_graph.nt...
Ad-hoc (Base): 40000 train / 5000 val / 5000 test cases
SULO: 40000 train / 5000 val / 5000 test cases

In [ ]:
def stream_and_build_factory(file_path):
    """Streams a large graph file line-by-line into a TriplesFactory (avoids
    loading the whole file into memory at once)."""
    import gc
    import array
    import numpy as np
    
    print(f"Streaming graph file {file_path} to save RAM...")
    entity_to_id = {}
    relation_to_id = {}
    flat_triples = array.array('i')  # packed int32s, far cheaper than a list of 3-element lists at this scale
    
    with open(file_path, "r", encoding="utf-8") as f:
        for count, line in enumerate(f):
            line = line.strip()
            if not line or line.startswith('#'): continue
            
            if '\t' in line:
                parts = line.split('\t')
            else:
                parts = line.split(' ', maxsplit=2)
                
            if len(parts) >= 3:
                h, r, o = parts[0].strip(), parts[1].strip(), parts[2].strip()
                
                if o.endswith(' .'): o = o[:-2]
                elif o.endswith('.'): o = o[:-1]
                
                if h not in entity_to_id: entity_to_id[h] = len(entity_to_id)
                if r not in relation_to_id: relation_to_id[r] = len(relation_to_id)
                if o not in entity_to_id: entity_to_id[o] = len(entity_to_id)
                
                flat_triples.append(entity_to_id[h])
                flat_triples.append(relation_to_id[r])
                flat_triples.append(entity_to_id[o])
                
            if count > 0 and count % 5000000 == 0:
                print(f"  ...processed {count} triples")

    print("Converting mapped triples to PyTorch tensors...")
    mapped_triples_array = np.frombuffer(flat_triples, dtype=np.int32).reshape(-1, 3)
    mapped_triples = torch.tensor(mapped_triples_array, dtype=torch.long)
    
    del flat_triples
    del mapped_triples_array
    gc.collect()

    print("Instantiating TriplesFactory...")
    return TriplesFactory(
        mapped_triples=mapped_triples,
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=False
    )

def train_pipeline(merged_path, emb_dim=16, lr=0.01, save_path=None, checkpoint_dir=None, checkpoint_name="rotate_checkpoint.pt", checkpoint_frequency=10, num_epochs=30, training_triples=None, batch_size=1024, sub_batch_size=1024, split=None, patience=15, eval_frequency=5):
    """Trains TransE embeddings. Returns (entity_mapping, entity_embeddings, stats).

    If `split` is given, num_epochs becomes a MAX budget: training runs in
    eval_frequency-epoch chunks, each scored by a quick RandomForestClassifier
    on split['val'] ROC-AUC, with early stopping after `patience` epochs of no
    improvement (returns the best checkpoint, not the last epoch). Chunks after
    the first rebuild the model/optimizer fresh and load the previous chunk's
    state_dict rather than using pykeen's continue_training (which requires a
    prior .train() call and isn't used here). If `split` is None, trains the
    full num_epochs in one call with no validation/early stopping.
    """
    if save_path and os.path.exists(save_path):
        print(f"Found existing embeddings at {save_path}. Skipping training...")
        # weights_only=False: checkpoint holds a raw numpy array + plain dict, not just tensors.
        checkpoint = torch.load(save_path, weights_only=False)
        stats = checkpoint.get('stats', {'runtime_seconds': None, 'epochs_trained': None, 'num_parameters': None, 'best_epoch': None, 'best_val_roc_auc': None})
        return checkpoint['mapping'], checkpoint['embeddings'], stats

    print("Parsing merged graph artifact into PyKEEN...")
    if not training_triples:
        training_triples = stream_and_build_factory(merged_path)

    gc.collect()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda':
        torch.cuda.empty_cache()

    print("Initiating TransE factorization training...")
    # scoring_fct_norm=1/margin=1.0 match Bordes et al. (2013)'s FB15k config.
    model = TransE(
        triples_factory=training_triples,
        embedding_dim=emb_dim,
        scoring_fct_norm=1,
        loss='marginranking',
        loss_kwargs=dict(margin=1.0),
        random_seed=42,
    ).to(device)

    num_parameters = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.Adam(params=model.get_grad_params(), lr=lr)

    training_loop = SLCWATrainingLoop(
        model=model,
        triples_factory=training_triples,
        optimizer=optimizer,
        negative_sampler=BasicNegativeSampler,
        negative_sampler_kwargs=dict(num_negs_per_pos=1),
    )

    # batch_size = optimizer.step() frequency; sub_batch_size = per-chunk memory footprint (decoupled since dense Adam touches the full param tensor every step).
    print(f"  batch_size={batch_size} (optimizer step frequency), "
          f"sub_batch_size={sub_batch_size} (memory footprint)")

    if checkpoint_dir:
        os.makedirs(checkpoint_dir, exist_ok=True)
        print(f"Checkpointing every {checkpoint_frequency} epochs to "
              f"{os.path.join(checkpoint_dir, checkpoint_name)}")

    entity_mapping = training_triples.entity_to_id

    def to_features(uris, labels):
        feats, targets = [], []
        for uri, y in zip(uris, labels):
            idx = entity_mapping.get(uri)
            if idx is None:
                continue  # case wasn't observed as a node in this KG's TransE graph
            feats.append(entity_embeddings_now[idx])
            targets.append(y)
        return np.asarray(feats), np.asarray(targets)

    # `losses` is PyKEEN's per-epoch training-loss list -- its length is the
    # ground truth for epochs actually completed (in case a run is ever
    # interrupted/resumed), rather than assuming num_epochs always ran in full.
    train_start = time.time()

    if split is None:
        train_kwargs = dict(
            triples_factory=training_triples,
            num_epochs=num_epochs,
            batch_size=batch_size,
            sub_batch_size=sub_batch_size,
        )
        if checkpoint_dir:
            train_kwargs.update(
                checkpoint_name=checkpoint_name,
                checkpoint_directory=checkpoint_dir,
                checkpoint_frequency=checkpoint_frequency,
                checkpoint_on_failure=True,  # also dump a checkpoint if an exception is raised mid-loop
            )
        try:
            losses = training_loop.train(**train_kwargs)
        except (RuntimeError, MemoryError):
            del training_loop, optimizer, model  # cleanup at point of failure so the caller's gc.collect() can actually reclaim GPU memory
            gc.collect()
            if device == 'cuda':
                torch.cuda.empty_cache()
            raise
        epochs_trained = len(losses) if losses is not None else num_epochs
        best_epoch = None
        best_val_roc_auc = None
        entity_embeddings = model.entity_representations[0](indices=None).detach().cpu().numpy()
    else:
        best_val_roc_auc = -1.0
        best_epoch = 0
        best_embeddings = None
        epochs_no_improve = 0
        epoch = 0
        prev_state = None
        prev_optimizer_state = None
        x_train_uris, y_train_labels = split['train']
        x_val_uris, y_val_labels = split['val']

        while epoch < num_epochs:
            chunk = min(eval_frequency, num_epochs - epoch)
            is_first_chunk = (epoch == 0)

            if not is_first_chunk:
                # Rebuild model/optimizer/loop each chunk, loading the previous
                # chunk's state before .train(continue_training=True) -- pykeen's
                # continue_training=False (the default) re-randomizes the model AND
                # replaces the optimizer on every .train() call, so a naive repeated
                # .train() call silently retrains from scratch each chunk.
                del training_loop, optimizer, model
                if device == 'cuda':
                    torch.cuda.empty_cache()
                gc.collect()
                model = TransE(
                    triples_factory=training_triples, embedding_dim=emb_dim,
                    scoring_fct_norm=1, loss='marginranking', loss_kwargs=dict(margin=1.0),
                    random_seed=42,
                ).to(device)
                model.load_state_dict(prev_state)
                optimizer = torch.optim.Adam(params=model.get_grad_params(), lr=lr)
                optimizer.load_state_dict(prev_optimizer_state)
                training_loop = SLCWATrainingLoop(
                    model=model, triples_factory=training_triples, optimizer=optimizer,
                    negative_sampler=BasicNegativeSampler,
                    negative_sampler_kwargs=dict(num_negs_per_pos=1),
                )

            try:
                losses = training_loop.train(
                    triples_factory=training_triples, num_epochs=chunk,
                    batch_size=batch_size, sub_batch_size=sub_batch_size,
                    continue_training=(not is_first_chunk),
                )
            except (RuntimeError, MemoryError):
                del training_loop, optimizer, model  # cleanup at point of failure, same reasoning as the split=None branch above

                gc.collect()
                if device == 'cuda':
                    torch.cuda.empty_cache()
                raise
            epoch += chunk
            avg_loss = float(np.mean(losses)) if losses else float('nan')
            # training_loop.optimizer (not the outer `optimizer` var) is the source of truth -- pykeen replaces it internally on the first chunk.
            prev_state = {k: v.detach().cpu().clone() for k, v in training_loop.model.state_dict().items()}
            prev_optimizer_state = training_loop.optimizer.state_dict()

            entity_embeddings_now = training_loop.model.entity_representations[0](indices=None).detach().cpu().numpy()
            x_train, y_train_feat = to_features(x_train_uris, y_train_labels)
            x_val, y_val_feat = to_features(x_val_uris, y_val_labels)

            clf = RandomForestClassifier(n_estimators=100, random_state=42)
            clf.fit(x_train, y_train_feat)
            val_probs = clf.predict_proba(x_val)[:, 1]
            val_roc_auc = roc_auc_score(y_val_feat, val_probs)
            del clf
            gc.collect()

            print(f"Epoch {epoch:03d} | Train Loss: {avg_loss:.4f} | Val ROC-AUC: {val_roc_auc:.4f}")

            if val_roc_auc > best_val_roc_auc:
                best_val_roc_auc = val_roc_auc
                best_epoch = epoch
                best_embeddings = entity_embeddings_now.copy()
                epochs_no_improve = 0
            else:
                epochs_no_improve += eval_frequency
                if epochs_no_improve >= patience:
                    print(f"Early stopping at epoch {epoch} (no Val ROC-AUC improvement in {patience} epochs).")
                    break

        epochs_trained = epoch
        entity_embeddings = best_embeddings

    runtime_seconds = time.time() - train_start

    stats = {
        'runtime_seconds': runtime_seconds,
        'epochs_trained': epochs_trained,
        'num_parameters': num_parameters,
        'best_epoch': best_epoch,
        'best_val_roc_auc': best_val_roc_auc,
    }
    print(f"Training finished in {runtime_seconds:.1f}s over {epochs_trained} epochs "
          f"({num_parameters:,} model parameters).")

    # Caller owns its own training_triples (e.g. baseline_triples/sulo_triples) and
    # must del/gc.collect() it separately -- this local binding doesn't free that.
    # optimizer is deleted alongside model/training_loop since Adam holds direct
    # references to model params + its own momentum/variance state (~doubles GPU footprint).
    del training_loop
    del model
    del optimizer
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

    if save_path:
        print(f"Saving extracted embeddings and mapping to {save_path}...")
        torch.save({'mapping': entity_mapping, 'embeddings': entity_embeddings, 'stats': stats}, save_path)

    return entity_mapping, entity_embeddings, stats

In [157]:
def classify(split, label, entity_mapping=None, entity_embeddings=None, pt_file_path=None):
    """Trains/evaluates a RandomForestClassifier on frozen TransE entity embeddings.
    `split` must be baseline_split/sulo_split -- the same train/val/test case
    URIs RGCN.ipynb/graph_sage.ipynb use, for a fair comparison."""
    if pt_file_path:
        if not os.path.exists(pt_file_path):
            raise FileNotFoundError(f"Error: Could not find the file {pt_file_path}")
        print(f"Loading embeddings directly from {pt_file_path}...")
        checkpoint = torch.load(pt_file_path, map_location='cpu', weights_only=False)
        entity_mapping = checkpoint['mapping']
        entity_embeddings = checkpoint['embeddings']
    elif entity_mapping is None or entity_embeddings is None:
        raise ValueError("You must provide either a 'pt_file_path' OR both 'entity_mapping' and 'entity_embeddings'.")

    def to_features(uris, labels):
        feats, targets = [], []
        for uri, y in zip(uris, labels):
            idx = entity_mapping.get(uri)
            if idx is None:
                continue  # case wasn't observed as a node in this KG's TransE graph
            feats.append(entity_embeddings[idx])
            targets.append(y)
        return np.asarray(feats), np.asarray(targets)

    x_train, y_train = to_features(*split['train'])
    x_test, y_test = to_features(*split['test'])

    train_total, test_total = len(split['train'][0]), len(split['test'][0])
    if len(x_train) < train_total or len(x_test) < test_total:
        print(f"WARNING [{label}]: only {len(x_train)}/{train_total} train and "
              f"{len(x_test)}/{test_total} test cases from the shared split were "
              f"found in this KG's TransE entity_mapping -- check for a masking/data mismatch.")

    print(f"[{label}] Training Random Forest on {len(x_train)} cases (same train split as RGCN/GraphSAGE)...")
    classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    fit_start = time.time()
    classifier.fit(x_train, y_train)
    predictions = classifier.predict(x_test)
    probabilities = classifier.predict_proba(x_test)[:, 1]
    runtime_seconds = time.time() - fit_start

    # Random Forest has no fixed weight count like a neural net -- total
    # decision-tree node count across the forest is its closest analog to a
    # "number of parameters" for the goals-doc record.
    num_parameters = sum(estimator.tree_.node_count for estimator in classifier.estimators_)

    print(f"[{label}] Evaluating on {len(x_test)} held-out test cases (same test split as RGCN/GraphSAGE)...")
    metrics = {
        'accuracy': accuracy_score(y_test, predictions),
        'precision': precision_score(y_test, predictions, zero_division=0),
        'recall': recall_score(y_test, predictions, zero_division=0),
        'f1': f1_score(y_test, predictions, zero_division=0),  # matches gnn_tools.evaluate_test_set's 'f1' methodology
        'class1_f1': f1_score(y_test, predictions, pos_label=1, zero_division=0),
        'class0_f1': f1_score(y_test, predictions, pos_label=0, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probabilities),
        'pr_auc': average_precision_score(y_test, probabilities),
        'num_train_cases': len(x_train),
        'num_test_cases': len(x_test),
        'runtime_seconds': runtime_seconds,
        'num_parameters': num_parameters,
    }
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    return classifier, metrics

In [158]:
def classify_logistic_regression(split, label, entity_mapping=None, entity_embeddings=None, pt_file_path=None):
    """Drop-in alternative to classify(): StandardScaler + LogisticRegression on
    the same frozen embeddings/split. Not called automatically -- call manually
    to compare classifier choices, and pass the result through
    build_result_row()/record_result() to persist it alongside the RandomForest rows."""
    if pt_file_path:
        if not os.path.exists(pt_file_path):
            raise FileNotFoundError(f"Error: Could not find the file {pt_file_path}")
        print(f"Loading embeddings directly from {pt_file_path}...")
        checkpoint = torch.load(pt_file_path, map_location='cpu', weights_only=False)
        entity_mapping = checkpoint['mapping']
        entity_embeddings = checkpoint['embeddings']
    elif entity_mapping is None or entity_embeddings is None:
        raise ValueError("You must provide either a 'pt_file_path' OR both 'entity_mapping' and 'entity_embeddings'.")

    def to_features(uris, labels):
        feats, targets = [], []
        for uri, y in zip(uris, labels):
            idx = entity_mapping.get(uri)
            if idx is None:
                continue  # case wasn't observed as a node in this KG's TransE graph
            feats.append(entity_embeddings[idx])
            targets.append(y)
        return np.asarray(feats), np.asarray(targets)

    x_train, y_train = to_features(*split['train'])
    x_test, y_test = to_features(*split['test'])

    train_total, test_total = len(split['train'][0]), len(split['test'][0])
    if len(x_train) < train_total or len(x_test) < test_total:
        print(f"WARNING [{label}]: only {len(x_train)}/{train_total} train and "
              f"{len(x_test)}/{test_total} test cases from the shared split were "
              f"found in this KG's TransE entity_mapping -- check for a masking/data mismatch.")

    print(f"[{label}] Training Logistic Regression on {len(x_train)} cases (same train split as RGCN/GraphSAGE)...")
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=42),
    )

    fit_start = time.time()
    classifier.fit(x_train, y_train)
    predictions = classifier.predict(x_test)
    probabilities = classifier.predict_proba(x_test)[:, 1]
    runtime_seconds = time.time() - fit_start

    fitted_lr = classifier.named_steps['logisticregression']
    num_parameters = fitted_lr.coef_.size + fitted_lr.intercept_.size

    print(f"[{label}] Evaluating on {len(x_test)} held-out test cases (same test split as RGCN/GraphSAGE)...")
    metrics = {
        'accuracy': accuracy_score(y_test, predictions),
        'precision': precision_score(y_test, predictions, zero_division=0),
        'recall': recall_score(y_test, predictions, zero_division=0),
        'f1': f1_score(y_test, predictions, zero_division=0),  # matches gnn_tools.evaluate_test_set's 'f1' methodology
        'class1_f1': f1_score(y_test, predictions, pos_label=1, zero_division=0),
        'class0_f1': f1_score(y_test, predictions, pos_label=0, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probabilities),
        'pr_auc': average_precision_score(y_test, probabilities),
        'num_train_cases': len(x_train),
        'num_test_cases': len(x_test),
        'runtime_seconds': runtime_seconds,
        'num_parameters': num_parameters,
    }
    for k, v in metrics.items():
        print(f"  {k}: {v}")

    return classifier, metrics

In [159]:
def build_result_row(kg_label, classifier_label, train_stats, metrics):
    """Flattens one KG's TransE-training stats + one classifier's metrics
    into a single result row, prefixed so same-named fields from the two
    dicts (e.g. both have 'runtime_seconds'/'num_parameters') don't collide."""
    row = {'kg': kg_label, 'classifier': classifier_label}
    row.update({f'embedding_{k}': v for k, v in train_stats.items()})
    row.update({f'classifier_{k}': v for k, v in metrics.items()})
    return row


def record_result(row, csv_path=transe_results_csv_path):
    """Appends one result row to csv_path (creating it with a header if needed),
    so results accumulate across cells/kernel sessions instead of overwriting."""
    df_new = pd.DataFrame([row])
    if os.path.exists(csv_path):
        df_combined = pd.concat([pd.read_csv(csv_path), df_new], ignore_index=True)
    else:
        df_combined = df_new
    df_combined.to_csv(csv_path, index=False)
    print(f"Recorded result to {csv_path} ({len(df_combined)} total rows)")
    return df_combined

In [ ]:
# Hyperparameter grids, swept one axis at a time (shaped like RGCN.ipynb's GRID_1A_DIM/GRID_2B_LR).
# emb_dim brackets Bordes et al. (2013)'s FB15k/WN18 optimal range (k=20-50);
# watch for CUDA OOM at emb_dim=50 on SULO's ~15M-entity table.
TRANSE_GRID_EMB_DIM = [16, 32, 50]
TRANSE_GRID_LR = [0.001, 0.005, 0.01, 0.05]

# Every config early-stops via train_pipeline's validation-based patience, so this
# is mostly a cap (not an exact epoch count) -- Phase 3's sweep below is close to
# a no-op as long as the cap is large enough for the self-selected best epoch.
TRANSE_GRID_EPOCHS = [100]

TRANSE_DEFAULT_EMB_DIM = 16
TRANSE_DEFAULT_LR = 0.01
TRANSE_DEFAULT_EPOCHS = 100


def transe_run_one_config(merged_path, training_triples, split, emb_dim, lr, num_epochs, save_path,
                           batch_size=1024, sub_batch_size=1024, patience=15, eval_frequency=5):
    """TransE analog of gnn_tools.run_one_config: trains one hyperparameter
    config via train_pipeline's early stopping, evaluated on split['val'].
    `training_triples` is built once by the caller and reused across the
    sweep. Mirrors run_one_config's OOM-safe try/except/finally handling."""
    start_time = time.time()
    status = 'ok'
    val_roc_auc = None
    best_epoch = None
    num_parameters = None
    epochs_trained = None

    try:
        entity_mapping, entity_embeddings, train_stats = train_pipeline(
            merged_path, training_triples=training_triples, emb_dim=emb_dim, lr=lr,
            save_path=save_path, num_epochs=num_epochs,
            batch_size=batch_size, sub_batch_size=sub_batch_size,
            split=split, patience=patience, eval_frequency=eval_frequency,
        )
        num_parameters = train_stats['num_parameters']
        val_roc_auc = train_stats.get('best_val_roc_auc')  # .get(): an old cached checkpoint may predate this key
        best_epoch = train_stats.get('best_epoch')
        epochs_trained = train_stats.get('epochs_trained')  # actual epochs run; num_epochs stays the requested max budget

        elapsed = time.time() - start_time
        auc_str = f"{val_roc_auc:.4f}" if val_roc_auc is not None else "N/A"
        print(f"  -> Finished in {elapsed:.1f}s | Val ROC-AUC: {auc_str} | "
              f"best_epoch={best_epoch}/{train_stats['epochs_trained']}")

    except (RuntimeError, MemoryError) as e:
        # Broadened to also catch pykeen's own sub_batch_size-search MemoryError, not just CUDA OOM.
        if isinstance(e, MemoryError) or "out of memory" in str(e).lower() or "does not fit in memory" in str(e).lower():
            print(f"  -> OOM/MemoryError: Skipping this configuration. ({e})")
            status = 'oom'
            for attr in ('last_traceback', 'last_type', 'last_value'):
                if hasattr(sys, attr):
                    delattr(sys, attr)
        else:
            raise e
        elapsed = time.time() - start_time
    finally:
        if 'entity_mapping' in locals():
            del entity_mapping
        if 'entity_embeddings' in locals():
            del entity_embeddings
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return {
        'emb_dim': emb_dim, 'lr': lr, 'num_epochs': num_epochs, 'best_epoch': best_epoch,
        'epochs_trained': epochs_trained,
        'val_roc_auc': val_roc_auc, 'num_parameters': num_parameters,
        'runtime_sec': elapsed, 'status': status,
    }

In [ ]:
def run_transe_sequential_tuning(
    schema_label, merged_path, training_triples, split,
    save_dir, csv_path,
    grid_emb_dim, grid_lr, grid_epochs,
    default_lr, default_epochs,
    batch_size=1024, sub_batch_size=1024,
    patience=15, eval_frequency=5,
):
    """TransE analog of gnn_tools.run_sequential_optimization_with_verification:
    greedy sequential sweep (emb_dim -> lr -> num_epochs), 2x2 verification over
    top candidates, one final held-out test evaluation. No depth/basis phases
    (no analog for TransE); the downstream classifier (RandomForest) is held
    fixed throughout. Every run early-stops via train_pipeline's validation
    ROC-AUC (patience/eval_frequency), so num_epochs is a max budget and the
    Phase 3 sweep mostly checks "is the cap big enough". Phase 5 refits on
    train+val combined before the single test evaluation, mirroring gnn_tools'
    combined_indices pattern. Returns (results_df, test_metrics, final_train_stats).
    """
    os.makedirs(f"{save_dir}robust_tuning_logs/", exist_ok=True)

    results = []
    phase_1_results = {}
    phase_3_results = {}

    def log_and_save(phase, parameter, val_roc_auc, elapsed, status, emb_dim, lr, num_epochs,
                      num_parameters=None, test_roc_auc=None, test_class0_f1=None, test_class1_f1=None,
                      best_epoch=None):
        row = {
            'phase': phase, 'parameter_tested': parameter,
            'val_roc_auc': val_roc_auc, 'test_roc_auc': test_roc_auc,
            'test_class0_f1': test_class0_f1, 'test_class1_f1': test_class1_f1,
            'num_parameters': num_parameters, 'runtime_sec': elapsed, 'status': status,
            'emb_dim': emb_dim, 'lr': lr, 'num_epochs': num_epochs, 'best_epoch': best_epoch,
        }
        results.append(row)
        pd.DataFrame(results).to_csv(csv_path, index=False)
        auc_str = f"{val_roc_auc:.4f}" if val_roc_auc is not None else "OOM/FAIL"
        print(f"  -> [{phase}] Val ROC-AUC: {auc_str} | Status: {status}")
        return row

    print("\n=======================================================")
    print(f" STARTING TRANSE SEQUENTIAL TUNING: {schema_label.upper()}")
    print("=======================================================")

    # ── PHASE 1: EMBEDDING DIMENSION SWEEP ───────────────────────────────────
    print(f"\n>>> Phase 1: Tuning emb_dim (lr={default_lr}, epochs={default_epochs})")
    best_dim = grid_emb_dim[0]
    best_dim_auc = -float('inf')
    phase_1_epochs = {}  # dim -> actual epochs_trained, for Phase 2's reuse branch below

    for dim in grid_emb_dim:
        print(f" [RUNNING] emb_dim: {dim}")
        row = transe_run_one_config(
            merged_path, training_triples, split, emb_dim=dim, lr=default_lr, num_epochs=default_epochs,
            save_path=f"{save_dir}robust_tuning_logs/{schema_label}_1_dim{dim}.pt",
            batch_size=batch_size, sub_batch_size=sub_batch_size,
            patience=patience, eval_frequency=eval_frequency,
        )
        log_and_save("1_EmbDim", f"dim_{dim}", row['val_roc_auc'], row['runtime_sec'], row['status'],
                     dim, default_lr, row['epochs_trained'], row['num_parameters'], best_epoch=row['best_epoch'])
        if row['status'] == 'ok' and row['val_roc_auc'] is not None:
            phase_1_results[dim] = row['val_roc_auc']
            phase_1_epochs[dim] = row['epochs_trained']
            if row['val_roc_auc'] > best_dim_auc:
                best_dim_auc = row['val_roc_auc']
                best_dim = dim

    sorted_dims = sorted(phase_1_results.items(), key=lambda x: x[1], reverse=True)
    top_dims = [dim for dim, auc in sorted_dims[:2]]
    if len(top_dims) == 1:
        top_dims.append(top_dims[0])
    print(f"\n>>> Phase 1 top emb_dims for verification: {top_dims}")
    gc.collect()

    # ── PHASE 2: LEARNING RATE SWEEP ─────────────────────────────────────────
    print(f"\n>>> Phase 2: Tuning lr (emb_dim locked at {best_dim})")
    best_lr = default_lr
    best_lr_auc = -float('inf')
    phase_2_epochs = {}  # lr -> actual epochs_trained, for Phase 3's reuse branch below

    for lr in grid_lr:
        if lr == default_lr:
            print(" [SKIPPING] Reusing Phase 1 result for default lr.")
            reused_epochs = phase_1_epochs.get(best_dim)
            log_and_save("2_LR", f"lr_{lr}", best_dim_auc, 0.0, "ok", best_dim, lr, reused_epochs)
            phase_2_epochs[lr] = reused_epochs
            if best_dim_auc > best_lr_auc:
                best_lr_auc = best_dim_auc
                best_lr = lr
            continue

        print(f" [RUNNING] lr: {lr}")
        row = transe_run_one_config(
            merged_path, training_triples, split, emb_dim=best_dim, lr=lr, num_epochs=default_epochs,
            save_path=f"{save_dir}robust_tuning_logs/{schema_label}_2_lr{lr}.pt",
            batch_size=batch_size, sub_batch_size=sub_batch_size,
            patience=patience, eval_frequency=eval_frequency,
        )
        log_and_save("2_LR", f"lr_{lr}", row['val_roc_auc'], row['runtime_sec'], row['status'],
                     best_dim, lr, row['epochs_trained'], row['num_parameters'], best_epoch=row['best_epoch'])
        phase_2_epochs[lr] = row['epochs_trained']
        if row['status'] == 'ok' and row['val_roc_auc'] is not None and row['val_roc_auc'] > best_lr_auc:
            best_lr_auc = row['val_roc_auc']
            best_lr = lr

    gc.collect()

    # ── PHASE 3: NUM_EPOCHS (MAX BUDGET) SWEEP ───────────────────────────────
    # See this function's docstring -- early stopping now makes each config
    # self-select its own best epoch, so this mostly checks whether the
    # budget cap is large enough rather than searching for the "right"
    # epoch count the way it did before early stopping existed.
    print(f"\n>>> Phase 3: Tuning num_epochs (emb_dim locked at {best_dim}, lr locked at {best_lr})")
    best_epochs = default_epochs
    best_epochs_auc = -float('inf')

    for n_epochs in grid_epochs:
        if n_epochs == default_epochs:
            print(" [SKIPPING] Reusing Phase 2 result for default num_epochs.")
            reused_epochs = phase_2_epochs.get(best_lr)
            log_and_save("3_Epochs", f"epochs_{n_epochs}", best_lr_auc, 0.0, "ok", best_dim, best_lr, reused_epochs)
            phase_3_results[(best_lr, n_epochs)] = best_lr_auc
            if best_lr_auc > best_epochs_auc:
                best_epochs_auc = best_lr_auc
                best_epochs = n_epochs
            continue

        print(f" [RUNNING] num_epochs: {n_epochs}")
        row = transe_run_one_config(
            merged_path, training_triples, split, emb_dim=best_dim, lr=best_lr, num_epochs=n_epochs,
            save_path=f"{save_dir}robust_tuning_logs/{schema_label}_3_epochs{n_epochs}.pt",
            batch_size=batch_size, sub_batch_size=sub_batch_size,
            patience=patience, eval_frequency=eval_frequency,
        )
        log_and_save("3_Epochs", f"epochs_{n_epochs}", row['val_roc_auc'], row['runtime_sec'], row['status'],
                     best_dim, best_lr, row['epochs_trained'], row['num_parameters'], best_epoch=row['best_epoch'])
        if row['status'] == 'ok' and row['val_roc_auc'] is not None:
            phase_3_results[(best_lr, n_epochs)] = row['val_roc_auc']
            if row['val_roc_auc'] > best_epochs_auc:
                best_epochs_auc = row['val_roc_auc']
                best_epochs = n_epochs

    sorted_reg = sorted(phase_3_results.items(), key=lambda x: x[1], reverse=True)
    top_reg = [params for params, auc in sorted_reg[:2]]
    if len(top_reg) == 1:
        top_reg.append(top_reg[0])
    print(f"\n>>> Phase 3 top (lr, num_epochs) combos for verification: {top_reg}")
    gc.collect()

    # ── PHASE 4: 2X2 VERIFICATION GRID ───────────────────────────────────────
    print("\n=======================================================")
    print(f" PHASE 4 (2X2 INTERACTION VERIFICATION): {schema_label.upper()}")
    print("=======================================================")

    # Deduplicate before attempting -- top_dims/top_reg can each contain a repeated
    # entry (from the "only one candidate" fallback above), and a failing (OOM) triple
    # would otherwise retry the identical guaranteed-to-fail attempt twice.
    verify_combos = []
    seen_combos = set()
    for verify_dim in top_dims:
        for verify_lr, verify_epochs in top_reg:
            key = (verify_dim, verify_lr, verify_epochs)
            if key in seen_combos:
                continue
            seen_combos.add(key)
            verify_combos.append(key)

    for verify_dim, verify_lr, verify_epochs in verify_combos:
        match_found = False
        for r in results:
            # Matched on (emb_dim, lr) alone -- epochs isn't independently varied here.
            if (r['emb_dim'] == verify_dim and r['lr'] == verify_lr and r['status'] == 'ok'):
                print(f" [SKIPPING] Cached: emb_dim={verify_dim} | lr={verify_lr} | epochs={verify_epochs}")
                log_and_save("4_Verification_Cached", f"dim_{verify_dim}_lr{verify_lr}_ep{verify_epochs}",
                             r['val_roc_auc'], 0.0, "ok", verify_dim, verify_lr, r['num_epochs'],
                             r['num_parameters'], best_epoch=r.get('best_epoch'))
                match_found = True
                break
        if match_found:
            continue

        print(f" [RUNNING Verification] emb_dim={verify_dim} | lr={verify_lr} | epochs={verify_epochs}")
        row = transe_run_one_config(
            merged_path, training_triples, split, emb_dim=verify_dim, lr=verify_lr, num_epochs=verify_epochs,
            save_path=(f"{save_dir}robust_tuning_logs/{schema_label}_verify_"
                       f"dim{verify_dim}_lr{verify_lr}_ep{verify_epochs}.pt"),
            batch_size=batch_size, sub_batch_size=sub_batch_size,
            patience=patience, eval_frequency=eval_frequency,
        )
        log_and_save("4_Verification", f"dim_{verify_dim}_lr{verify_lr}_ep{verify_epochs}",
                     row['val_roc_auc'], row['runtime_sec'], row['status'],
                     verify_dim, verify_lr, row['epochs_trained'], row['num_parameters'], best_epoch=row['best_epoch'])

    phase4_runs = [r for r in results if r['status'] == 'ok' and r['phase'] in ('4_Verification', '4_Verification_Cached')]
    ok_results = [r for r in results if r['status'] == 'ok']
    absolute_winner = max(phase4_runs, key=lambda x: x['val_roc_auc']) if phase4_runs else max(ok_results, key=lambda x: x['val_roc_auc'])

    print("\n=======================================================")
    print(f" TUNING COMPLETE FOR {schema_label.upper()}")
    print(f" Best Phase 4 val ROC-AUC: {absolute_winner['val_roc_auc']:.4f}")
    print(f" Optimal settings: emb_dim={absolute_winner['emb_dim']} | lr={absolute_winner['lr']} | "
          f"num_epochs={absolute_winner['num_epochs']}")
    print("=======================================================")
    gc.collect()

    # ── PHASE 5: FINAL GENERALIZATION TEST (train+val refit, test evaluated once) ──
    print("\n=======================================================")
    print(f" PHASE 5: FINAL GENERALIZATION TEST -- {schema_label.upper()}")
    print("=======================================================")

    winning_dim = absolute_winner['emb_dim']
    winning_lr = absolute_winner['lr']
    final_epoch_budget = default_epochs  # full budget, not the search run's actual convergence epoch

    final_embeddings_path = f"{save_dir}{schema_label}_final_transe_embeddings.pt"
    try:
        entity_mapping, entity_embeddings, final_train_stats = train_pipeline(
            merged_path, training_triples=training_triples, emb_dim=winning_dim, lr=winning_lr,
            save_path=final_embeddings_path, num_epochs=final_epoch_budget,
            batch_size=batch_size, sub_batch_size=sub_batch_size,
            split=split, patience=patience, eval_frequency=eval_frequency,
        )
    except (RuntimeError, MemoryError) as e:
        print(f"\n>>> PHASE 5 FAILED for {schema_label}: {e}")
        print(">>> Phases 1-4 already found a winning config (see above/the tuning "
              "CSV), but it could not be retrained for the final held-out test -- "
              "GPU memory may be more constrained now than earlier in this run.")
        log_and_save("5_Final_Test_Evaluation", "unified_model", absolute_winner['val_roc_auc'],
                     0.0, "failed", winning_dim, winning_lr, None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return pd.DataFrame(results), None, None

    def to_features(uris, labels):
        feats, targets = [], []
        for uri, y in zip(uris, labels):
            idx = entity_mapping.get(uri)
            if idx is None:
                continue
            feats.append(entity_embeddings[idx])
            targets.append(y)
        return np.asarray(feats), np.asarray(targets)

    # Refit on train+val combined -- mirrors gnn_tools' Phase 4 combined_indices
    # pattern, so TransE's final number uses the same effective training-data
    # fraction the GNNs' final models do, rather than leaving val idle.
    trainval_uris = split['train'][0] + split['val'][0]
    trainval_labels = np.concatenate([split['train'][1], split['val'][1]])
    x_trainval, y_trainval = to_features(trainval_uris, trainval_labels)
    x_test, y_test = to_features(*split['test'])

    fit_start = time.time()
    final_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    final_classifier.fit(x_trainval, y_trainval)
    test_predictions = final_classifier.predict(x_test)
    test_probabilities = final_classifier.predict_proba(x_test)[:, 1]
    test_runtime_seconds = time.time() - fit_start

    test_metrics = {
        'accuracy': accuracy_score(y_test, test_predictions),
        'precision': precision_score(y_test, test_predictions, zero_division=0),
        'recall': recall_score(y_test, test_predictions, zero_division=0),
        'f1': f1_score(y_test, test_predictions, zero_division=0),  # matches gnn_tools.evaluate_test_set's 'f1' methodology for cross-model comparability
        'class1_f1': f1_score(y_test, test_predictions, pos_label=1, zero_division=0),
        'class0_f1': f1_score(y_test, test_predictions, pos_label=0, zero_division=0),
        'roc_auc': roc_auc_score(y_test, test_probabilities),
        'pr_auc': average_precision_score(y_test, test_probabilities),
        'num_train_cases': len(x_trainval),
        'num_test_cases': len(x_test),
        'runtime_seconds': test_runtime_seconds,
        'num_parameters': sum(estimator.tree_.node_count for estimator in final_classifier.estimators_),
    }

    log_and_save(
        "5_Final_Test_Evaluation", "unified_model", absolute_winner['val_roc_auc'], 0.0, "ok",
        winning_dim, winning_lr, final_train_stats.get('epochs_trained'),
        num_parameters=final_train_stats['num_parameters'],
        test_roc_auc=test_metrics['roc_auc'],
        test_class0_f1=test_metrics['class0_f1'],
        test_class1_f1=test_metrics['class1_f1'],
        best_epoch=final_train_stats.get('best_epoch'),
    )

    print("\n=======================================================")
    print(f" {schema_label.upper()} FINAL GENERALIZATION REPORT")
    print("=======================================================")
    for k, v in test_metrics.items():
        print(f" * Test {k}: {v}")
    print("=======================================================")

    del entity_mapping, entity_embeddings, final_classifier
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(results), test_metrics, final_train_stats


## Train Baseline

In [17]:
if os.path.exists(baseline_embeddings_path):
    print(f"Found cached embeddings at {baseline_embeddings_path} -- skipping TriplesFactory construction.")
    baseline_triples = None
else:
    baseline_triples = stream_and_build_factory(baseline_merged_path)

Streaming graph file /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_baseline_training.nt to save RAM...
Converting mapped triples to PyTorch tensors...
Instantiating TriplesFactory...

In [ ]:
# batch_size matched to the SULO cell below (must be identical across KGs for a fair comparison).
# split=baseline_split enables early stopping -- num_epochs becomes a max budget, not an exact count.
baseline_mapping, baseline_embeddings, baseline_train_stats = train_pipeline(baseline_merged_path, training_triples=baseline_triples, emb_dim=64, save_path=baseline_embeddings_path, checkpoint_dir=f"{volume}transe_checkpoint", checkpoint_name="baseline_checkpoint.pt", batch_size=8192, sub_batch_size=1024, split=baseline_split, eval_frequency=5)

del baseline_triples  # train_pipeline's own del only drops its local binding, not this global
gc.collect()

In [19]:
_, baseline_metrics = classify(baseline_split, "Ad-hoc (Base)", entity_mapping=baseline_mapping, entity_embeddings=baseline_embeddings)

del baseline_mapping, baseline_embeddings  # free now that classify() has consumed them
gc.collect()

[Ad-hoc (Base)] Training Random Forest on 40000 cases (same train split as RGCN/GraphSAGE)...
[Ad-hoc (Base)] Evaluating on 5000 held-out test cases (same test split as RGCN/GraphSAGE)...
  accuracy: 0.5074
  precision: 0.5112240576027107
  recall: 0.4797297297297297
  f1_class1: 0.494976419930285
  f1_class0: 0.519227015420652
  roc_auc: 0.5108134029169835
  pr_auc: 0.5098582325779526
  num_train_cases: 40000
  num_test_cases: 5000
  runtime_seconds: 80.4593677520752
  num_parameters: 1171968
373

## Train SULO

In [ ]:
if os.path.exists(sulo_embeddings_path):
    print(f"Found cached embeddings at {sulo_embeddings_path} -- skipping TriplesFactory construction.")
    sulo_triples = None
else:
    sulo_triples = stream_and_build_factory(sulo_merged_path)

In [ ]:
# split=sulo_split enables early stopping -- num_epochs becomes a max budget, not an exact count.
sulo_mapping, sulo_embeddings, sulo_train_stats = train_pipeline(sulo_merged_path, save_path=sulo_embeddings_path, emb_dim=32, checkpoint_dir=f"{volume}transe_checkpoint", checkpoint_name="sulo_checkpoint.pt", training_triples=sulo_triples, batch_size=8192, sub_batch_size=1024, split=sulo_split, eval_frequency=5)

del sulo_triples
gc.collect()

In [ ]:
_, sulo_metrics = classify(sulo_split, "SULO", entity_mapping=sulo_mapping, entity_embeddings=sulo_embeddings)

del sulo_mapping, sulo_embeddings
gc.collect()

## Save Output

In [51]:
print("\n=== TransE + RandomForest summary (same case split as RGCN/GraphSAGE) ===")
for name, train_stats, m in [
    ('Ad-hoc (Base)', baseline_train_stats, baseline_metrics),
    ('SULO', sulo_train_stats, sulo_metrics),
]:
    print(f"\n{name}:")
    print("  -- TransE embedding training --")
    for k, v in train_stats.items():
        print(f"  {k}: {v}")
    print("  -- Downstream classifier --")
    for k, v in m.items():
        print(f"  {k}: {v}")

record_result(build_result_row('Ad-hoc (Base)', 'RandomForest', baseline_train_stats, baseline_metrics))
record_result(build_result_row('SULO', 'RandomForest', sulo_train_stats, sulo_metrics))

TypeError: Cannot read properties of undefined (reading 'status')

## Hyperparameter Tuning (RQ2a -- mirrors RGCN.ipynb's sequential tuning)

In [ ]:
# Unlike "Train Baseline" above, a sweep needs the TriplesFactory built unconditionally
# (reused across many emb_dim/lr/num_epochs configs, so no single-file cache short-circuit).
baseline_triples = stream_and_build_factory(baseline_merged_path)

baseline_tuning_df, baseline_tuned_metrics, baseline_tuned_train_stats = run_transe_sequential_tuning(
    schema_label='base', merged_path=baseline_merged_path, training_triples=baseline_triples,
    split=baseline_split, save_dir=transe_out_dir,
    csv_path=f"{transe_out_dir}base_sequential_tuning_results.csv",
    grid_emb_dim=TRANSE_GRID_EMB_DIM, grid_lr=TRANSE_GRID_LR, grid_epochs=TRANSE_GRID_EPOCHS,
    default_lr=TRANSE_DEFAULT_LR, default_epochs=TRANSE_DEFAULT_EPOCHS,
    batch_size=16384, sub_batch_size=4096,  # matched to the SULO cell below for a fair comparison
)

if baseline_tuned_metrics is not None:
    record_result(build_result_row('Ad-hoc (Base)', 'RandomForest (tuned)', baseline_tuned_train_stats, baseline_tuned_metrics))
else:
    print('Phase 5 failed for Ad-hoc (Base) -- see the printed failure above and '
          "base_sequential_tuning_results.csv's final row (status=failed). "
          'Skipping record_result rather than crashing on None metrics.')

del baseline_triples
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
sulo_triples = stream_and_build_factory(sulo_merged_path)

sulo_tuning_df, sulo_tuned_metrics, sulo_tuned_train_stats = run_transe_sequential_tuning(
    schema_label='sulo', merged_path=sulo_merged_path, training_triples=sulo_triples,
    split=sulo_split, save_dir=transe_out_dir,
    csv_path=f"{transe_out_dir}sulo_sequential_tuning_results.csv",
    grid_emb_dim=TRANSE_GRID_EMB_DIM, grid_lr=TRANSE_GRID_LR, grid_epochs=TRANSE_GRID_EPOCHS,
    default_lr=TRANSE_DEFAULT_LR, default_epochs=TRANSE_DEFAULT_EPOCHS,
    # batch_size raised (not sub_batch_size) since SULO's ~15M-entity table makes
    # optimizer.step() frequency, not raw triple volume, the dominant wall-clock cost.
    batch_size=16384, sub_batch_size=4096,
)

if sulo_tuned_metrics is not None:
    record_result(build_result_row('SULO', 'RandomForest (tuned)', sulo_tuned_train_stats, sulo_tuned_metrics))
else:
    print('Phase 5 failed for SULO -- see the printed failure above and '
          "sulo_sequential_tuning_results.csv's final row (status=failed). "
          'Skipping record_result rather than crashing on None metrics.')

del sulo_triples
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Diagnostics — why is SULO tuning so slow?

Run cells 0-3 first (imports + path setup), not cells 4+ (case split, embedding
training, tuning sweep). Safest to run in a **second notebook attached to the
same cluster** so it doesn't wait behind or interrupt a live SULO sweep.

In [61]:
# 1. Is a GPU actually available/being used right now? nvidia-smi shows node-wide
# usage (needed if the live run is a different process/notebook); torch.cuda.* only
# reflects this process's own CUDA context.
import subprocess

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device name: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  total memory: {props.total_memory / (1024**3):.1f} GB")
    print(f"  allocated (this process only): {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    print(f"  reserved (this process only): {torch.cuda.memory_reserved(0) / (1024**3):.2f} GB")

print("\nnvidia-smi (node-wide, all processes):")
try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,name',
         '--format=csv'],
        capture_output=True, text=True, timeout=10,
    )
    print(out.stdout or out.stderr)
    print("If utilization.gpu is near 0% while the SULO sweep is supposedly running, "
          "training has silently fallen back to CPU or is stalled on something else "
          "(I/O, a Python-side loop) rather than actually computing on the GPU.")
except Exception as e:
    print(f"  nvidia-smi not available in this environment: {e}")

print("\nnvidia-smi (per-process, this GPU):")
try:
    out = subprocess.run(
        ['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv'],
        capture_output=True, text=True, timeout=10,
    )
    print(out.stdout or out.stderr)
except Exception as e:
    print(f"  not available: {e}")

torch.cuda.is_available(): True
  device name: Tesla T4
  total memory: 15.6 GB
  allocated (this process only): 0.00 GB
  reserved (this process only): 0.00 GB

nvidia-smi (node-wide, all processes):
utilization.gpu [%], memory.used [MiB], memory.total [MiB], name
0 %, 3 MiB, 16384 MiB, Tesla T4

If utilization.gpu is near 0% while the SULO sweep is supposedly running, training has silently fallen back to CPU or is stalled on something else (I/O, a Python-side loop) rather than actually computing on the GPU.

nvidia-smi (per-process, this GPU):
pid, used_gpu_memory [MiB]

In [62]:
# 2. Raw file-size / line-count comparison between the two merged training files.
def fast_line_count(path):
    n = 0
    t0 = time.time()
    with open(path, 'rb') as f:
        for _ in f:
            n += 1
    return n, time.time() - t0

line_counts = {}
for _label, _path in [('Ad-hoc (Base)', baseline_merged_path), ('SULO', sulo_merged_path)]:
    if not os.path.exists(_path):
        print(f"[missing] {_label}: {_path}")
        continue
    size_gb = os.path.getsize(_path) / (1024 ** 3)
    n_lines, elapsed = fast_line_count(_path)
    line_counts[_label] = n_lines
    print(f"{_label}: {_path}")
    print(f"  size on disk: {size_gb:.2f} GB")
    print(f"  lines: {n_lines:,} (counted in {elapsed:.1f}s)")

if 'Ad-hoc (Base)' in line_counts and 'SULO' in line_counts and line_counts['Ad-hoc (Base)'] > 0:
    ratio = line_counts['SULO'] / line_counts['Ad-hoc (Base)']
    print(f"\nSULO has {ratio:.1f}x as many lines as Ad-hoc (Base).")

Ad-hoc (Base): /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_baseline_training.nt
  size on disk: 0.53 GB
  lines: 1,920,797 (counted in 1.3s)
SULO: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_sulo_training.nt
  size on disk: 10.63 GB
  lines: 39,460,128 (counted in 24.2s)

SULO has 20.5x as many lines as Ad-hoc (Base).

In [63]:
# 3. Entity/relation dedup ratio (sampled, not full-file) -- is SULO bigger because of
# genuinely more triples over a similar vocabulary, or near-1:1 entity explosion?
def sample_entity_stats(path, sample_lines=500_000):
    entity_seen, relation_seen = set(), set()
    n = 0
    t0 = time.time()
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t') if '\t' in line else line.split(' ', maxsplit=2)
            if len(parts) < 3:
                continue
            h, r, o = parts[0].strip(), parts[1].strip(), parts[2].strip()
            entity_seen.add(h)
            entity_seen.add(o)
            relation_seen.add(r)
            n += 1
            if n >= sample_lines:
                break
    elapsed = time.time() - t0
    dedup_ratio = len(entity_seen) / (2 * n) if n else float('nan')
    return {
        'sampled_lines': n, 'elapsed_sec': elapsed,
        'unique_entities': len(entity_seen), 'unique_relations': len(relation_seen),
        'entity_dedup_ratio': dedup_ratio,  # 1.0 = zero reuse, every (h,o) slot is a new entity
    }


dedup_stats = {}
for _label, _path in [('Ad-hoc (Base)', baseline_merged_path), ('SULO', sulo_merged_path)]:
    if not os.path.exists(_path):
        continue
    stats = sample_entity_stats(_path)
    dedup_stats[_label] = stats
    print(f"{_label} (first {stats['sampled_lines']:,} lines, {stats['elapsed_sec']:.1f}s):")
    print(f"  unique entities in sample: {stats['unique_entities']:,} "
          f"(dedup ratio {stats['entity_dedup_ratio']:.3f})")
    print(f"  unique relations in sample: {stats['unique_relations']:,}\n")

Ad-hoc (Base) (first 500,000 lines, 1.3s):
  unique entities in sample: 359,641 (dedup ratio 0.360)
  unique relations in sample: 18

SULO (first 500,000 lines, 3.1s):
  unique entities in sample: 499,535 (dedup ratio 0.500)
  unique relations in sample: 17

In [64]:
# 4. Micro-benchmark: train TransE on a small prefix of the real file, measure triples/sec.
import tempfile


def make_prefix_file(src_path, n_lines):
    tmp_path = tempfile.NamedTemporaryFile(delete=False, suffix='.nt').name
    with open(src_path, 'r', encoding='utf-8') as fin, open(tmp_path, 'w', encoding='utf-8') as fout:
        for i, line in enumerate(fin):
            if i >= n_lines:
                break
            fout.write(line)
    return tmp_path


def micro_benchmark(label, merged_path, n_lines=300_000, emb_dim=8, bench_epochs=1, batch_size=1024):
    print(f"\n=== Micro-benchmark: {label} ({n_lines:,}-line prefix, emb_dim={emb_dim}) ===")
    prefix_path = make_prefix_file(merged_path, n_lines)
    model = optimizer = loop = factory = None
    try:
        t0 = time.time()
        factory = stream_and_build_factory(prefix_path)
        build_elapsed = time.time() - t0
        n_entities = len(factory.entity_to_id)
        n_relations = len(factory.relation_to_id)
        n_triples = factory.mapped_triples.shape[0]
        print(f"  built TriplesFactory in {build_elapsed:.1f}s -- "
              f"{n_entities:,} entities, {n_relations:,} relations, {n_triples:,} triples")

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model = TransE(triples_factory=factory, embedding_dim=emb_dim, random_seed=42).to(device)
        optimizer = torch.optim.Adam(params=model.get_grad_params(), lr=0.01)
        loop = SLCWATrainingLoop(
            model=model, triples_factory=factory, optimizer=optimizer,
            negative_sampler=BasicNegativeSampler, negative_sampler_kwargs=dict(num_negs_per_pos=1),
        )

        t1 = time.time()
        loop.train(
            triples_factory=factory, num_epochs=bench_epochs,
            batch_size=batch_size, sub_batch_size=batch_size, use_tqdm=False,
        )
        train_elapsed = time.time() - t1

        triples_per_sec = (n_triples * bench_epochs) / train_elapsed if train_elapsed > 0 else float('nan')
        print(f"  trained {bench_epochs} epoch(s) over {n_triples:,} triples in "
              f"{train_elapsed:.1f}s -> {triples_per_sec:,.0f} triples/sec on {device}")

        return {
            'n_entities': n_entities, 'n_relations': n_relations, 'n_triples_sample': n_triples,
            'triples_per_sec': triples_per_sec, 'device': device, 'build_elapsed_sec': build_elapsed,
        }
    finally:
        for obj in (model, optimizer, loop, factory):
            del obj
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        os.remove(prefix_path)


bench_results = {}
bench_results['Ad-hoc (Base)'] = micro_benchmark('Ad-hoc (Base)', baseline_merged_path)
bench_results['SULO'] = micro_benchmark('SULO', sulo_merged_path)

=== Micro-benchmark: Ad-hoc (Base) (300,000-line prefix, emb_dim=8) ===
Streaming graph file /tmp/tmp1rxxtwf0.nt to save RAM...
Converting mapped triples to PyTorch tensors...
Instantiating TriplesFactory...
  built TriplesFactory in 1.2s -- 193,107 entities, 17 relations, 300,000 triples
  trained 1 epoch(s) over 300,000 triples in 2.0s -> 150,806 triples/sec on cuda

=== Micro-benchmark: SULO (300,000-line prefix, emb_dim=8) ===
Streaming graph file /tmp/tmpa0peihao.nt to save RAM...
Converting mapped triples to PyTorch tensors...
Instantiating TriplesFactory...
  built TriplesFactory in 1.1s -- 299,535 entities, 17 relations, 300,000 triples
  trained 1 epoch(s) over 300,000 triples in 1.9s -> 159,228 triples/sec on cuda

In [65]:
# 5. Synthesis: does the D4 throughput explain the observed wall-clock, projected
# from D2's line count?
OBSERVED_RUNTIME_SEC = {
    # Paste in what you're actually seeing in results.csv / the live cell,
    # e.g. the dim_8 row's runtime_sec for a *_num_epochs_config total, or
    # a wall-clock estimate of how long the current cell has been running.
    'SULO_1_EmbDim_dim_8_20_epochs': 47790.67,
}

for _label in ('Ad-hoc (Base)', 'SULO'):
    if _label not in bench_results or _label not in line_counts:
        continue
    b = bench_results[_label]
    total_lines = line_counts[_label]
    triples_per_sec = b['triples_per_sec']
    if not triples_per_sec or triples_per_sec != triples_per_sec:  # NaN/0 guard
        continue

    projected_epoch_sec = total_lines / triples_per_sec
    print(f"{_label}: ~{total_lines:,} total triples / {triples_per_sec:,.0f} triples/sec "
          f"(measured) -> projected ~{projected_epoch_sec / 60:.1f} min/epoch, "
          f"~{projected_epoch_sec * 20 / 3600:.2f} hours for a 20-epoch run.")

print(f"\nCompare that projection against what's actually observed, e.g.:")
for _run_label, _observed in OBSERVED_RUNTIME_SEC.items():
    print(f"  {_run_label}: {_observed / 3600:.2f} hours actually taken")

print(
    "\nIf the projected time is close to what's observed: the SULO run isn't "
    "'broken', it's just genuinely this much bigger than Ad-hoc, and the fix "
    "is a scale change (bigger batch_size, fewer epochs in the grid, a "
    "smaller entity vocabulary, or accepting the runtime).\n"
    "If the projection is far SHORTER than what's observed (e.g. minutes vs "
    "hours): something in the real run is different from this benchmark -- "
    "most likely candidates, in order of likelihood given the code as it "
    "stands today:\n"
    "  1. No GPU engagement in the real run (see the nvidia-smi cell above) "
    "-- CPU fallback would be 1-2 orders of magnitude slower than the GPU "
    "number this benchmark just measured.\n"
    "  2. GPU contention -- another job/notebook sharing the same cluster's "
    "single GPU.\n"
    "  3. I/O stalls reading merged_sulo_training.nt itself from the mounted "
    "/Volumes/ path (network filesystem) during stream_and_build_factory, "
    "which runs once per transe_run_one_config() call in the sweep (it "
    "re-parses the merged file from scratch for the *first* config, then "
    "reuses the in-memory `training_triples` for the rest -- see cell "
    "'Hyperparameter tuning: SULO' above)."
)

Ad-hoc (Base): ~1,920,797 total triples / 150,806 triples/sec (measured) -> projected ~0.2 min/epoch, ~0.07 hours for a 20-epoch run.
SULO: ~39,460,128 total triples / 159,228 triples/sec (measured) -> projected ~4.1 min/epoch, ~1.38 hours for a 20-epoch run.

Compare that projection against what's actually observed, e.g.:
  SULO_1_EmbDim_dim_8_20_epochs: 13.28 hours actually taken

If the projected time is close to what's observed: the SULO run isn't 'broken', it's just genuinely this much bigger than Ad-hoc, and the fix is a scale change (bigger batch_size, fewer epochs in the grid, a smaller entity vocabulary, or accepting the runtime).
If the projection is far SHORTER than what's observed (e.g. minutes vs hours): something in the real run is different from this benchmark -- most likely candidates, in order of likelihood given the code as it stands today:
  1. No GPU engagement in the real run (see the nvidia-smi cell above) -- CPU fallback would be 1-2 orders of magnitude slower th

In [66]:
# 6. Isolate table-size cost from data-volume cost: full-scale vocab, same 300k-triple
# sample as D4 -- a throughput drop here would point to optimizer.step()'s full-table
# cost (dense Adam), not raw data volume.
def micro_benchmark_full_vocab(label, merged_path, sample_triples=300_000, emb_dim=8,
                                bench_epochs=1, batch_size=1024):
    print(f"\n=== Full-vocab micro-benchmark: {label} ===")
    t0 = time.time()
    full_factory = stream_and_build_factory(merged_path)
    build_elapsed = time.time() - t0
    n_entities = len(full_factory.entity_to_id)
    n_relations = len(full_factory.relation_to_id)
    n_total_triples = full_factory.mapped_triples.shape[0]
    print(f"  built FULL TriplesFactory in {build_elapsed:.1f}s -- "
          f"{n_entities:,} entities, {n_relations:,} relations, {n_total_triples:,} triples")

    g = torch.Generator().manual_seed(42)
    idx = torch.randperm(n_total_triples, generator=g)[:sample_triples]
    sub_factory = TriplesFactory(
        mapped_triples=full_factory.mapped_triples[idx],
        entity_to_id=full_factory.entity_to_id,
        relation_to_id=full_factory.relation_to_id,
        create_inverse_triples=False,
    )

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = TransE(triples_factory=sub_factory, embedding_dim=emb_dim, random_seed=42).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.Adam(params=model.get_grad_params(), lr=0.01)
    loop = SLCWATrainingLoop(
        model=model, triples_factory=sub_factory, optimizer=optimizer,
        negative_sampler=BasicNegativeSampler, negative_sampler_kwargs=dict(num_negs_per_pos=1),
    )

    t1 = time.time()
    loop.train(
        triples_factory=sub_factory, num_epochs=bench_epochs,
        batch_size=batch_size, sub_batch_size=batch_size, use_tqdm=False,
    )
    train_elapsed = time.time() - t1

    triples_per_sec = (sample_triples * bench_epochs) / train_elapsed if train_elapsed > 0 else float('nan')
    print(f"  model has {n_params:,} parameters (full vocab)")
    print(f"  trained {bench_epochs} epoch(s) over {sample_triples:,} SAMPLED triples "
          f"(same count as the earlier small-vocab benchmark) in {train_elapsed:.1f}s "
          f"-> {triples_per_sec:,.0f} triples/sec on {device}")

    del model, optimizer, loop, sub_factory, full_factory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'n_entities': n_entities, 'n_relations': n_relations, 'n_params': n_params,
        'triples_per_sec': triples_per_sec, 'device': device, 'sample_triples': sample_triples,
    }


full_vocab_bench = micro_benchmark_full_vocab('SULO', sulo_merged_path)

print(f"\nEarlier small-vocab benchmark: {bench_results['SULO']['triples_per_sec']:,.0f} triples/sec "
      f"with {bench_results['SULO']['n_entities']:,} entities.")
print(f"This full-vocab benchmark:     {full_vocab_bench['triples_per_sec']:,.0f} triples/sec "
      f"with {full_vocab_bench['n_entities']:,} entities -- same {full_vocab_bench['sample_triples']:,} triples trained.")
_slowdown = (bench_results['SULO']['triples_per_sec'] / full_vocab_bench['triples_per_sec']
             if full_vocab_bench['triples_per_sec'] else float('nan'))
print(f"Slowdown factor from table size alone: {_slowdown:.1f}x")
print(
    "If this slowdown factor is roughly in the same ballpark as the 9.5x gap between the "
    "earlier naive projection (1.4h) and the actual observed runtime (13.28h), that confirms "
    "the embedding-table size (not data volume) is what's driving the wall-clock, and the fix "
    "is decoupling batch_size (optimizer step frequency) from sub_batch_size (memory footprint) "
    "in train_pipeline -- not more epochs, not less data."
)

=== Full-vocab micro-benchmark: SULO ===
Streaming graph file /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_sulo_training.nt to save RAM...
  ...processed 5000000 triples
  ...processed 10000000 triples
  ...processed 15000000 triples
  ...processed 20000000 triples
  ...processed 25000000 triples
  ...processed 30000000 triples
  ...processed 35000000 triples
Converting mapped triples to PyTorch tensors...
Instantiating TriplesFactory...
  built FULL TriplesFactory in 118.4s -- 15,010,314 entities, 24 relations, 39,460,128 triples
  model has 120,082,704 parameters (full vocab)
  trained 1 epoch(s) over 300,000 SAMPLED triples (same count as the earlier small-vocab benchmark) in 18.8s -> 15,921 triples/sec on cuda

Earlier small-vocab benchmark: 159,228 triples/sec with 299,535 entities.
This full-vocab benchmark:     15,921 triples/sec with 15,010,314 entities -- same 300,000 triples trained.
Slowdown factor from table size alone: 10.0x
If this slowdown factor is r

In [151]:
_smoke_test_triples = stream_and_build_factory(sulo_merged_path)

Streaming graph file /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/merged_sulo_training.nt to save RAM...
  ...processed 5000000 triples
  ...processed 10000000 triples
  ...processed 15000000 triples
  ...processed 20000000 triples
  ...processed 25000000 triples
  ...processed 30000000 triples
  ...processed 35000000 triples
Converting mapped triples to PyTorch tensors...
Instantiating TriplesFactory...

In [ ]:
# Smoke test on real full-scale SULO data for 1 epoch -- fails fast (seconds) on OOM
# instead of hours in. Needs cells 1-10 run fresh first. save_path=None: nothing persisted.
_smoke_start = time.time()
_smoke_result = transe_run_one_config(
    sulo_merged_path, _smoke_test_triples, sulo_split,
    emb_dim=32, lr=0.01, num_epochs=1,
    save_path=None,
    batch_size=16384, sub_batch_size=8192,
)
_smoke_elapsed = time.time() - _smoke_start 

print(f"\nSmoke test result: {_smoke_result}")
print(f"Total smoke test wall-clock (factory build + 1 epoch + RF val fit): {_smoke_elapsed / 60:.1f} min")
print(
    "No OOM and this completed -> safe to launch the full sweep. Rough sanity check: "
    "subtract the ~2 min factory-build time, and per-epoch time here times 20 epochs "
    "times ~13 configs in the grid gives a ballpark for the full Phase 1-4 sweep -- "
    "compare against the ~13.28h the old batch_size=1024 config took for just ONE "
    "20-epoch run to see the improvement."
)

del _smoke_test_triples
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Parsing merged graph artifact into PyKEEN...
Initiating TransE factorization training...
  batch_size=16384 (optimizer step frequency), sub_batch_size=8192 (memory footprint)
The training loop just failed during epoch 1 due to error CUDA out of memory. Tried to allocate 1.79 GiB. GPU 0 has a total capacity of 15.56 GiB of which 1014.62 MiB is free. Process 1253600 has 7.28 GiB memory in use. Process 1257912 has 7.28 GiB memory in use. Of the allocated memory 7.16 GiB is allocated by PyTorch, and 8.73 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables).
  -> CUDA OOM: Skipping this configuration.

Smoke test result: {'emb_dim': 32, 'lr': 0.01, 'num_epochs': 1, 'val_roc_auc': None, 'num_parameters': None, 'runtime_sec': 17.24185609817505, 'status': 'oom'}
T